# Day 7 · Plotly EDA 1차: 현재 위험 구조 파악

**목적:** 최신 제공 연도의 `산업중분류 × 사업장 규모`별 사고 구조를 살펴보고, 이후 챗봇 답변과 Streamlit 화면에서 재사용할 **정리된 데이터(`eda`)와 Plotly 그래프 함수**를 만든다.

이 노트북은 CSV만 읽습니다. DB나 LLM 호출은 없으며, WSL 프로젝트의 `.venv` 커널에서 **Run All**하면 표와 그래프가 차례로 나옵니다.


## 1. 기존 파일과 전처리 방식

Day 1/2처럼 `data/`의 CSV를 `utf-8-sig`로 읽습니다. 4개 파일 모두 `대업종`·`구분(산업중분류)` 뒤에 규모별 값 **10열**이 있습니다. 규모 열 제목은 파일마다 달라서 Day 2의 **열 순서**를 공통 규모 이름으로 맞춥니다.

| 원본 파일의 지표 | 이 노트북의 열 | 의미 |
| --- | --- | --- |
| 사업장수 | `사업장수` | 해당 산업·규모의 사업장 수 |
| 사고재해자수 | `사고재해자수` | 사고로 재해를 입은 사람 수 |
| 사고사망자수 | `사고사망자수` | 사고로 사망한 사람 수 |
| 사망만인율 | `사망만인율` | 원본이 제공한 상대 지표 |

파일명 끝의 날짜가 가장 늦은 **공통 스냅샷**을 사용합니다. 현재 제공 파일은 `20251231`입니다.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

# Day 2와 같은 방식으로 노트북 폴더/저장소 루트 어느 쪽에서 실행해도 data를 찾습니다.
cwd = Path.cwd().resolve()
candidates = [cwd, *(parent / 'src/rag_project_1/Project_1' for parent in (cwd, *cwd.parents))]
PROJECT_DIR = next(p for p in candidates if p.name == 'Project_1' and (p / 'data').is_dir())
DATA_DIR = PROJECT_DIR / 'data'

METRICS = ('사업장수', '사고재해자수', '사고사망자수', '사망만인율')
SIZE_ORDER = ['5인 미만', '5-9인', '10-19인', '20-29인', '30-49인',
              '50-99인', '100-299인', '300-499인', '500-999인', '1000인 이상']
reference_date = max(p.stem.rsplit('_', 1)[-1]
                     for p in DATA_DIR.glob('*산업중분류별 규모별 사업장수_*.csv'))
reference_year = reference_date[:4]
paths = {metric: next(DATA_DIR.glob(f'*산업중분류별 규모별 {metric}_{reference_date}.csv'))
         for metric in METRICS}
print('자료 기준일:', f'{reference_date[:4]}-{reference_date[4:6]}-{reference_date[6:]}')
display(pd.DataFrame({'지표': list(paths), '파일': [path.name for path in paths.values()]}))


## 2. 원본 구조와 결측 확인

원본의 `구분` 값은 산업중분류로 사용합니다. 먼저 행·열 수, 10개 규모 열, 결측 개수를 확인합니다.


In [ ]:
stats = {metric: pd.read_csv(path, encoding='utf-8-sig') for metric, path in paths.items()}
source_check = pd.DataFrame([
    {'지표': metric, '행': len(raw), '규모 열': len(raw.columns) - 2,
     '산업중분류 수': raw['구분'].nunique(), '결측 칸': int(raw.isna().sum().sum())}
    for metric, raw in stats.items()
])
display(source_check)
for metric, raw in stats.items():
    print(f'{metric} 규모 열:', list(raw.columns[2:]))


## 3. 챗봇·대시보드용 정리 데이터

**정규화 각주:** 4개 파일의 규모 열 이름은 다르지만 순서가 같으므로 3~12번째 열을 공통 `SIZE_ORDER`에 대응시킵니다. 각 파일을 긴 형태로 바꾼 뒤 `대업종 + 산업중분류 + 규모`로 결합합니다. `사망만인율`의 빈칸은 0으로 채우지 않습니다.


In [ ]:
long_tables = {}
for metric, raw in stats.items():
    normalized = raw.copy()
    normalized.columns = ['대업종', '산업중분류', *SIZE_ORDER]
    long_tables[metric] = normalized.melt(
        id_vars=['대업종', '산업중분류'], value_vars=SIZE_ORDER,
        var_name='규모', value_name=metric,
    )

eda = long_tables['사업장수']
for metric in METRICS[1:]:
    eda = eda.merge(long_tables[metric], on=['대업종', '산업중분류', '규모'],
                    validate='one_to_one')
eda['규모'] = pd.Categorical(eda['규모'], categories=SIZE_ORDER, ordered=True)
eda = eda.sort_values(['산업중분류', '규모']).reset_index(drop=True)
eda.insert(0, '기준연도', int(reference_year))
display(eda.head(10))
print('정리된 행:', len(eda), '| 산업중분류:', eda['산업중분류'].nunique(),
      '| 규모:', eda['규모'].nunique())
print('규모 값:', SIZE_ORDER)
display(eda[list(METRICS)].isna().sum().rename('결측 칸').to_frame())


### 수치 해석 주의

`사망만인율`은 원본의 제공값을 그대로 사용합니다. 근로자 수 분모가 이 4개 파일에 없으므로 `사업장수`나 `사고사망자수`만으로 다시 계산하지 않습니다. 아래처럼 두 원본 지표가 단순히 일치하지 않는 조합도 있어, 챗봇에서 원인까지 단정하면 안 됩니다.


In [ ]:
rate_gap = eda[(eda['사고사망자수'].eq(0)) & (eda['사망만인율'].gt(0))]
print('사고사망자수 0명인데 사망만인율 > 0인 조합:', len(rate_gap))
display(rate_gap[['산업중분류', '규모', '사고사망자수', '사망만인율']].head(5))


## 4. Streamlit에서도 쓸 Plotly 함수

두 함수 모두 정리된 `eda`를 입력받아 **Figure를 반환**합니다. 노트북에서는 `.show()`로 표시하고, 이후 Streamlit에서는 같은 Figure를 `st.plotly_chart(...)`에 전달하면 됩니다.


In [ ]:
# 각주 1: 모든 Heatmap의 산업 행을 사고사망자수 합계 순으로 고정해 지표 간 위치 비교를 쉽게 합니다.
# color_cap은 극단적인 사망만인율이 다른 칸의 색을 모두 눌러 버릴 때만 사용하며, hover에는 원값이 남습니다.
def plot_heatmap(data: pd.DataFrame, metric: str, color_cap: float | None = None):
    industry_order = (data.groupby('산업중분류', observed=True)['사고사망자수']
                      .sum().sort_values(ascending=False).index)
    matrix = (data.pivot(index='산업중분류', columns='규모', values=metric)
              .reindex(index=industry_order, columns=SIZE_ORDER))
    fig = px.imshow(matrix, aspect='auto', zmin=0, zmax=color_cap,
                    color_continuous_scale='YlOrRd',
                    labels={'x': '사업장 규모', 'y': '산업중분류', 'color': metric},
                    title=f'{reference_year}년 산업중분류 × 규모: {metric}', height=850)
    fig.update_layout(margin=dict(l=260, r=40, t=65, b=75))
    return fig


# 각주 2: 선택한 규모에서 해당 지표가 큰 산업중분류만 추려 가로 막대로 그립니다.
# 사망만인율처럼 값 차이가 매우 크면 log_x=True로 눈금만 로그로 바꾸고 hover 값은 원값을 유지합니다.
def plot_industry_bar(data: pd.DataFrame, size: str, metric: str,
                      top_n: int = 15, log_x: bool = False):
    ranked = (data.loc[data['규모'].eq(size)].dropna(subset=[metric])
              .nlargest(top_n, metric).sort_values(metric))
    fig = px.bar(ranked, x=metric, y='산업중분류', orientation='h',
                 hover_data=['대업종', '사업장수', '사고재해자수', '사고사망자수', '사망만인율'],
                 title=f'{reference_year}년 {size}: 산업중분류별 {metric} 상위 {top_n}개',
                 height=650)
    fig.update_traces(marker_color='#2563eb')
    fig.update_layout(margin=dict(l=260, r=45, t=65, b=65), yaxis_title=None)
    if log_x:
        fig.update_xaxes(type='log', title=f'{metric} (로그 눈금)')
    return fig


## 5. Heatmap: 사고재해자수

산업과 규모를 한눈에 보고, 사고가 많이 기록된 **조합**을 찾습니다. 셀에 마우스를 올리면 원값을 확인할 수 있습니다.


In [ ]:
plot_heatmap(eda, '사고재해자수').show()
top_injury = eda.nlargest(1, '사고재해자수').iloc[0]
display(Markdown(
    f"**챗봇 질문:** 어떤 산업·규모에서 사고재해자가 가장 많은가?  \n**핵심 인사이트:** {top_injury['산업중분류']} · {top_injury['규모']}에서 "
    f"{top_injury['사고재해자수']:,.0f}명으로 가장 많다. 이는 절대 인원이다."
))


## 6. Heatmap: 사고사망자수

사고재해자수와 별도로 사망자수의 집중 구간을 봅니다.


In [ ]:
plot_heatmap(eda, '사고사망자수').show()
top_death = eda.nlargest(1, '사고사망자수').iloc[0]
display(Markdown(
    f"**챗봇 질문:** 어느 산업·규모에서 사고사망자수가 가장 많은가?  \n**핵심 인사이트:** {top_death['산업중분류']} · {top_death['규모']}에서 "
    f"{top_death['사고사망자수']:,.0f}명으로 가장 많다."
))


## 7. Heatmap: 사망만인율

극단값 때문에 색이 한쪽으로 몰리지 않도록 **색상 상한만 95백분위수**로 설정합니다. 실제 사망만인율은 hover에서 원값으로 보입니다. 빈칸은 자료가 없는 조합입니다.


In [ ]:
rate_color_cap = eda['사망만인율'].quantile(0.95)
plot_heatmap(eda, '사망만인율', color_cap=rate_color_cap).show()
top_rate = eda.nlargest(1, '사망만인율').iloc[0]
display(Markdown(
    f"**챗봇 질문:** 어느 산업·규모에서 원본 사망만인율이 가장 높은가?  \n**핵심 인사이트:** {top_rate['산업중분류']} · {top_rate['규모']}의 "
    f"제공값은 {top_rate['사망만인율']:,.2f}이다. 절대 사망자수와는 별도로 해석한다."
))


## 8. 선택 규모의 산업별 Bar Chart

기본 규모는 `5인 미만`입니다. 다른 규모를 보고 싶다면 아래 `selected_size`만 바꿔 다시 실행하면 됩니다. **Run All에는 변경이 필요 없습니다.**


In [ ]:
selected_size = '5인 미만'
selected = eda.loc[eda['규모'].eq(selected_size)]
print('선택 규모:', selected_size)
plot_industry_bar(eda, selected_size, '사고사망자수').show()
top_selected_death = selected.nlargest(1, '사고사망자수').iloc[0]
display(Markdown(
    f"**챗봇 질문:** {selected_size} 사업장에서는 어느 산업의 사고사망자수가 많은가?  \n**핵심 인사이트:** {top_selected_death['산업중분류']}가 "
    f"{top_selected_death['사고사망자수']:,.0f}명으로 가장 많다."
))


### 같은 규모의 사망만인율 Bar Chart

원값 차이가 매우 커서 **가로축만 로그 눈금**으로 표시합니다. 막대 hover에는 원본 수치가 그대로 나타납니다.


In [ ]:
plot_industry_bar(eda, selected_size, '사망만인율', log_x=True).show()
top_selected_rate = selected.nlargest(1, '사망만인율').iloc[0]
display(Markdown(
    f"**챗봇 질문:** {selected_size}에서 사고사망자수 1위와 사망만인율 1위는 같은가?  \n**핵심 인사이트:** 사망자수 1위는 {top_selected_death['산업중분류']}, "
    f"사망만인율 1위는 {top_selected_rate['산업중분류']}이다. "
    f"{'같다' if top_selected_death['산업중분류'] == top_selected_rate['산업중분류'] else '다르다'}."
))


## 9. 챗봇용 짧은 비교 근거

다음 표는 두 핵심 질문에 바로 사용할 수 있는 요약입니다. 순위는 이 파일의 산업중분류·규모 조합만 대상으로 하며, 사망만인율 자료의 분모와 정의는 원자료 설명을 추가 확인해야 합니다.


In [ ]:
death_top5 = selected.nlargest(5, '사고사망자수')['산업중분류'].tolist()
rate_top5 = selected.nlargest(5, '사망만인율')['산업중분류'].tolist()
insights = pd.DataFrame([
    {'질문': '어떤 산업·규모에서 사고가 많은가?',
     '짧은 답변': f"사고재해자수는 {top_injury['산업중분류']} · {top_injury['규모']} "
                 f"({top_injury['사고재해자수']:,.0f}명), 사고사망자수는 "
                 f"{top_death['산업중분류']} · {top_death['규모']} "
                 f"({top_death['사고사망자수']:,.0f}명)가 가장 많다."},
    {'질문': '사고사망자수가 많은 곳과 사망만인율이 높은 곳은 같은가?',
     '짧은 답변': f"{selected_size} 기준 사망자수 1위는 {death_top5[0]}, "
                 f"사망만인율 1위는 {rate_top5[0]}. "
                 f"상위 5개 산업의 교집합은 {len(set(death_top5) & set(rate_top5))}개다."},
])
display(insights)


### 다음 단계에서 재사용할 것

- `eda`: 1행이 `기준연도 × 산업중분류 × 규모`인 정리 데이터.
- `plot_heatmap(eda, metric)`, `plot_industry_bar(eda, size, metric)`: Plotly Figure를 반환하는 함수.
- `insights`: 챗봇의 첫 질문 후보와 데이터에 근거한 짧은 답변.

이 수치는 산업·규모의 **현황**을 설명합니다. 사업장수만을 분모로 위험률을 새로 만들거나, 높은 값의 원인을 이 그래프만으로 단정하지 않습니다.
